# Build the upxBuilder Android APK on Google Colab

No PC setup needed — Colab provides the Linux machine, Java and the Android SDK are installed below.

**How to use:** menu **Runtime → Run all**, wait ~10–15 minutes, and the last cell hands you `app-debug.apk`. Then copy it to your phone and tap it to install (allow "Install unknown apps").

In [ ]:
%%bash
# ── Step 1: Java 17 + Android SDK ──────────────────────────────────────
set -e
echo '==> Installing Java 17…'
apt-get -qq update >/dev/null
apt-get -qq install -y openjdk-17-jdk-headless >/dev/null

echo '==> Downloading Android command-line tools…'
mkdir -p /content/android-sdk/cmdline-tools
cd /content/android-sdk/cmdline-tools
rm -rf latest cmdline-tools tools.zip
wget -q -O tools.zip https://dl.google.com/android/repository/commandlinetools-linux-11076708_latest.zip
unzip -q tools.zip && rm tools.zip
mv cmdline-tools latest

echo '==> Installing SDK components (platform 34, build-tools, NDK 26.1, CMake 3.22)…'
yes | latest/bin/sdkmanager --licenses >/dev/null 2>&1 || true
latest/bin/sdkmanager --install 'platform-tools' 'platforms;android-34' 'build-tools;34.0.0' 'ndk;26.1.10909125' 'cmake;3.22.1' >/dev/null
echo 'Setup done.'

In [ ]:
%%bash
# ── Step 2: clone the repo and build the APK ───────────────────────────
set -e
export ANDROID_HOME=/content/android-sdk
export JAVA_HOME=/usr/lib/jvm/java-17-openjdk-amd64

cd /content
rm -rf New-work
echo '==> Cloning DanyFox12/New-work…'
git clone -q https://github.com/DanyFox12/New-work.git
cd New-work
git checkout -q claude/practical-faraday-fd4wyi

cd upxBuilder-android
echo "sdk.dir=$ANDROID_HOME" > local.properties
chmod +x gradlew
echo '==> Building (first run downloads Gradle + dependencies — be patient)…'
./gradlew --no-daemon assembleDebug
echo
echo '==> APK ready:'
ls -lh app/build/outputs/apk/debug/app-debug.apk

In [ ]:
# ── Step 3: download the APK to this computer ──────────────────────────
from google.colab import files
files.download('/content/New-work/upxBuilder-android/app/build/outputs/apk/debug/app-debug.apk')

### Optional: save to Google Drive instead

Easier when you build from a phone browser — the APK appears in the Google Drive app on your phone; tap it there to install.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!cp /content/New-work/upxBuilder-android/app/build/outputs/apk/debug/app-debug.apk /content/drive/MyDrive/upxBuilder.apk
print('Saved to Drive as upxBuilder.apk — open the Google Drive app on your phone and tap it to install.')

## After installing on the phone

1. Open **upxBuilder** (phone needs internet).
2. Tap **Setup** (download icon in the toolbar / welcome screen) → **Install core dev set**.
3. Add big SDKs when needed, from Setup or the Terminal tab:
   ```
   pkg install flutter           # Flutter + Dart (~1 GB, use Wi-Fi)
   pkg install sdk               # Android sdkmanager
   pkg install platform-tools    # adb & fastboot
   pkg help                      # everything available
   ```
4. Create a project (New Project → Flutter / C++ / Java / Kotlin / Python / JS / Go) and press **Run**.